In [7]:
from time import time
from pop import Oled, Potentiometer, Switches, Leds, delay

class Display():
    file = open("./StopWatch_pals.txt", "w+")
    oled = Oled(automode = False)
    name = None
    
    def __init__(self, name = None):
        if name:
            self.name = name
    
        self.initScreen()
        
    def __del__(self):
        display = self.oled
        display.clearDisplay()
        display.display()
        
        self.file.close()
        
    def initScreen(self):
        display = self.oled
        
        display.clearDisplay()
        display.setTextSize(2)
        display.setCursor(0, 0)
        
        display.print(("%s" % self.name).center(10))
        
        display.setTextSize(1)
        display.setCursor(0, 24)
        
        display.print("SW1 : Start  / Stop\n")
        display.print("SW2 : Lap    / Reset")
        
        display.display()
        
    def mainScreen(self, isStop):
        display = self.oled
        
        display.fillRect(0, 16, 128, 48, display.BLACK)
        
        display.setTextSize(1)
        display.setCursor(14, 20)
        
        display.print("Laps")
        
        display.drawRect(0, 18, 52, 46, display.WHITE)
        
        sw1 = "Start" if isStop else "Stop"
        sw2 = "Reset" if isStop else "Laps"
        
        display.setCursor(56, 22)
        display.print("SW1 : %s" % sw1)
        
        display.setCursor(56, 38)
        display.print("SW2 : %s" % sw2)
        
        display.setCursor(56, 52)
        display.print("<-Scroll Lap")
        
        display.display()
            
    def timeScreen(self, time):
        display = self.oled
        
        display.fillRect(0, 0, 128, 15, display.BLACK)
        
        display.setTextSize(1)
        display.setCursor(0, 0)
        
        m_second = time % 100
        second = time // 100 % 60
        minute = time // 6000
        
        string = "%02d:%02d.%02d"%(minute, second, m_second)
        
        display.setCursor(0, 0)
        display.setTextSize(2)
        
        display.print(string.center(10))
        
        display.display()
        
    def lapsScreen(self, strings, scroll):
        v_axis = [30, 42, 54]
        display = self.oled
        
        display.fillRect(1, 27, 49, 36, display.BLACK)
        display.setTextSize(1)
        
        for i, string in enumerate(strings[scroll * 3:]):
            if i is 3:
                break
            display.setCursor(1, y_axis[i])
            display.print(string)
            
        display.drawLine(49, 30 + (scroll -1)*3, 49, 30 + scroll * 3, display.WHITE)
        
        display.display()
        
def display_test():
    ds = Display("Test")
    delay(1000)
    ds.mainScreen(True)
    delay(1000)
    ds.timeScreen(100 * 30)
    delay(1000)
    ds.mainScreen(False)
    delay(1000)
    
    strings = ["00:00:00", "11:11:11", "22:22:22", "33:33:33", "44:44:44", "55:55:55"]
    ds.lapsScreen(strings, 0)
    delay(1000)
    ds.lapsScreen(strings, 1)
    delay(1000)
    ds.mainScreen(True)
    
class StopWatch(Display):
    STOP = 1
    START = 0
    
    init_flag = False
    
    def __init__(self):
        self._status = self.STOP
        
        self._time = 0
        self._start = 0
        self.laps = []
        
        self.potentiometer = Potentiometer()
        
        Display.__init__(self, self.__class__.__name__)
        
    def start(self):
        if not self.init_flag:
            self._start = time() * 100
            self.init_flag = True
        self._status = self.START
        
    def stop(self):
        self._status = self.STOP
    
    def toggle(self):
        if self._status is self.STOP:
            self.start()
        else:
            self.stop()
        self.mainScreen(self._status)
    
    def reset(self):
        if self._status is self.STOP:
            self.init_flag = False
            self._start = time() * 100
            self._time = 0
            self.laps = []
            
            self.timeScreen(0)
            self.lapsScreen([], 0)
        else:
            print("COMMAND ERROR")
            
    def lap(self):
        if self._status is self.START:
            if len(self.laps) >= 11 * 3:
                self.laps = self.laps[1:11*3]
            
            ms = self._time % 100
            sec = self._time // 100 % 60
            min = self._time // 6000
            
            string = "%02d:%02d.%02d" % (min, sec, ms)
            self.laps.append(string)
            self.file.writelines(string)
            
        else:
            print("COMMAND ERROR")
            
    def mode(self):
        if self._status is self.STOP:
            self.reset()
        else:
            self.lap()
            
    def update(self):
        if self._time >= 360_000:
            self.reset()
        
        if self._status == self.START:
            self._time = time() * 100 - self._start
            self.timeScreen(self._time)
        else:
            self._start = time() * 100 - self._time
            self._time = time() * 100 - self._start
            
        if self.init_flag:
            scroll = round(self.potentiometer.read() / 4096 *10)
            self.lapsScreen(self.laps, scroll)
            
if __name__ == "__main__":
    stopWatch = StopWatch()
    switches = Switches()
    
    sw1 = switches[0].read
    sw2 = switches[1].read
    
    while sw1() or sw2():
        if not sw1():
            while not sw1(): pass
            stopWatch.toggle()
        elif not sw2():
            while not sw2(): pass
            stopWatch.mode()
        
        stopWatch.update()
        delay(1)
        
stopWatch.oled.clearDisplay()
stopWatch.oled.display()

        
        
        
            
            
            

KeyboardInterrupt: 

In [2]:
clear()